### Problem 001: Network Delay Time (LeetCode 743)

### Problem Definition and Constraints
You are given a network of `n` directed nodes, labeled from `1` to `n`. 
You are given `times`, a list of directed edges where `times[i] = (u, v, t)`:
* `u` is the source node.
* `v` is the target node.
* `t` is the time it takes for a signal to travel from `u` to `v`.

You start by sending a signal from node `k`. 
Return the minimum time it takes for *all* `n` nodes to receive the signal. If it is impossible for all the nodes to receive the signal (e.g., someone is disconnected), return `-1`.

**Examples:**
* **Example 1:**
  * **Input:** `times = [[1,2,1],[2,3,1],[1,4,4],[3,4,1]], n = 4, k = 1`
  * **Output:** `3`
* **Example 2:**
  * **Input:** `times = [[1,2,1],[2,3,1]], n = 3, k = 2`
  * **Output:** `-1` (Node 1 can never be reached).

* Constraints:
  * 1 <= k <= n <= 100
  * 1 <= times.length <= 1000

### Dijkstra's Algorithm (Min-Heap) Approach
If this were an unweighted graph, we would use a standard BFS Queue. But because edges have different times, a path with 3 short jumps might be faster than a path with 1 massive jump. A standard Queue just blindly processes "1 jump away, 2 jumps away," which fails here.

**The Solution:** We replace the standard Queue with a **Min-Heap (Priority Queue)**. 
A Min-Heap automatically sorts everything inside it so that the item with the *lowest total time* is always at the very front. 
1. We start at node `k` at time `0`.
2. We pull the fastest available route out of the Min-Heap.
3. If we haven't visited that node yet, we lock it in (add to `visited` set). The time it took to get here is officially the fastest possible time.
4. We look at its neighbors, add the current time to the neighbor's travel time, and throw them into the Min-Heap.
5. We repeat until the Heap is empty. If we visited `n` unique nodes, the time it took to reach the very last node is our answer.

* **Time Complexity:** $O(E \log V)$ — $E$ is the number of edges and $V$ is the number of vertices (nodes). In the worst case, we push every single edge into the Min-Heap. Pushing to a heap takes logarithmic time.
* **Space Complexity:** $O(V + E)$ — To store the Adjacency List (which holds every node and edge) and the Min-Heap itself.

In [7]:
import collections
import heapq
from typing import List

class Solution:
    def networkDelayTime(self, times: List[List[int]], n: int, k: int) -> int:
        # Step 1: Build the Adjacency List
        # Dictionary mapping: node -> list of (travel_time, neighbor_node)
        adj = collections.defaultdict(list)
        for u, v, t in times:
            adj[u].append((t, v))
            
        # Step 2: Initialize the Min-Heap and Visited Set
        # The heap stores tuples of (accumulated_time, node)
        # We start at node k with 0 accumulated time.
        min_heap = [(0, k)] 
        visited = set()
        
        # We will track the time it takes to reach the furthest node
        total_time = 0
        
        # Step 3: Dijkstra's Algorithm
        while min_heap:
            # heappop ALWAYS gives us the node with the lowest accumulated time
            time, node = heapq.heappop(min_heap)
            
            # THE BRAKES: If we already found a faster way to this node, skip it.
            if node in visited:
                continue
                
            # Lock in this node as visited
            visited.add(node)
            
            # Update our master clock to the time it took to reach this node
            total_time = max(total_time, time)
            
            # Step 4: Explore the neighbors
            for neighbor_time, neighbor in adj[node]:
                if neighbor not in visited:
                    # Accumulate the time: (time it took to get here + time to neighbor)
                    heapq.heappush(min_heap, (time + neighbor_time, neighbor))
                    
        # Step 5: Did the signal reach everyone?
        if len(visited) == n:
            return total_time
        else:
            return -1

### Problem 002: Min Cost to Connect Points (LeetCode 1584)

### Problem Definition and Constraints
You are given a 2-D integer array `points`, where `points[i] = [xi, yi]`. Each `points[i]` represents a distinct point on a 2-D plane.
The cost of connecting two points `[xi, yi]` and `[xj, yj]` is the **Manhattan distance** between the two points: `|xi - xj| + |yi - yj|`.

Return the minimum cost to connect all points together, such that there exists exactly one path between each pair of points. 

**Examples:**
* **Example 1:**
  * **Input:** `points = [[0,0],[2,2],[3,3],[2,4],[4,2]]`
  * **Output:** `10`

* Constraints:
  * 1 <= points.length <= 1000
  * -1,000,000 <= xi, yi <= 1,000,000
  * All pairs (xi, yi) are distinct.

### Minimum Spanning Tree (MST) & Kruskal's Algorithm
This problem is asking for a **Minimum Spanning Tree (MST)**. 
* **Spanning:** It must connect every single point together.
* **Tree:** There are no loops (cycles). 
* **Minimum:** It uses the absolute cheapest total cost to do it.

To solve this, we use **Kruskal's Algorithm** paired with a tool called **Union-Find (Disjoint Set Union - DSU)**. 

**The Strategy:**
1. **The Blueprint:** We calculate the cost (Manhattan distance) of building a bridge between *every single possible pair* of points on the board.
2. **The Sorting:** We take all those potential bridges and sort them from cheapest to most expensive.
3. **The Build:** We start buying the cheapest bridges one by one. 
4. **The Rule (Union-Find):** Before we build a bridge, we ask: *"Are these two points already connected by a different path?"* If they are, building this bridge would create a useless loop (a cycle), so we throw it away. If they aren't, we build it and add the cost to our total.
5. We stop when we have built exactly $n - 1$ bridges (because it takes exactly 4 bridges to connect 5 points).

**The Math & Complexity Breakdown:**
* **Time Complexity:** $O(n^2 \log n)$ 
  * Calculating every possible edge takes $O(n^2)$ time.
  * Sorting those $n^2$ edges takes $O(n^2 \log(n^2))$, which mathematically simplifies to $O(n^2 \log n)$. 
  * The Union-Find operations take nearly $O(1)$ time each, so sorting is the bottleneck.
* **Space Complexity:** $O(n^2)$ — We have to store every single possible edge in a list before we sort it. For $n$ points, there are roughly $n^2 / 2$ edges.

In [8]:
from typing import List

# Helper Class: Union-Find (Disjoint Set)
class UnionFind:
    def __init__(self, n):
        # Initially, every point is its own boss (parent)
        self.parent = list(range(n))
        # Rank helps keep the tree flat when we merge groups
        self.rank = [1] * n

    def find(self, i):
        # Find the absolute top boss of the group
        if self.parent[i] != i:
            # Path compression: point directly to the top boss
            self.parent[i] = self.find(self.parent[i])
        return self.parent[i]

    def union(self, i, j):
        # Find the bosses of both points
        p1, p2 = self.find(i), self.find(j)
        
        # If they have the same boss, they are already connected! (Cycle detected)
        if p1 == p2:
            return False
            
        # If they have different bosses, merge them based on rank
        if self.rank[p1] > self.rank[p2]:
            self.parent[p2] = p1
        elif self.rank[p1] < self.rank[p2]:
            self.parent[p1] = p2
        else:
            self.parent[p2] = p1
            self.rank[p1] += 1
            
        return True

class Solution:
    def minCostConnectPoints(self, points: List[List[int]]) -> int:
        n = len(points)
        edges = []
        
        # Step 1: Generate all possible edges (The Blueprint)
        for i in range(n):
            for j in range(i + 1, n):
                x1, y1 = points[i]
                x2, y2 = points[j]
                # Manhattan distance formula
                dist = abs(x1 - x2) + abs(y1 - y2)
                
                # Store as (cost, point1_index, point2_index)
                edges.append((dist, i, j))
                
        # Step 2: Sort edges from cheapest to most expensive
        edges.sort()
        
        # Step 3: Kruskal's Algorithm using Union-Find
        uf = UnionFind(n)
        total_cost = 0
        edges_used = 0
        
        for dist, i, j in edges:
            # Try to connect point i and point j
            if uf.union(i, j): 
                # If True, they weren't connected yet. Build the bridge!
                total_cost += dist
                edges_used += 1
                
                # A tree connecting n points ALWAYS has exactly n - 1 edges
                if edges_used == n - 1:
                    break
                    
        return total_cost

### Problem 003: Cheapest Flights Within K Stops (LeetCode 787)

### Problem Definition and Constraints
You are given `n` airports, labeled `0` to `n - 1`. 
You are given a list of `flights` where `flights[i] = [source, destination, price]`. 
You want to fly from `src` to `dst`. 

Find the **cheapest** total price, but you are only allowed to make at most **`k` stops**. (Note: `k` stops means taking at most `k + 1` actual flights). If it is impossible, return `-1`.

**Examples:**
* **Example 1:**
  * **Input:** `n = 4, flights = [[0,1,200],[1,2,100],[1,3,300],[2,3,100]], src = 0, dst = 3, k = 1`
  * **Output:** `500`
  * *Explanation:* Path `0 -> 1 -> 2 -> 3` costs 400, but it has 2 stops. We are only allowed 1 stop. The path `0 -> 1 -> 3` costs 500 and only has 1 stop, so it is the winner.

* Constraints:
  * 1 <= n <= 100
  * 1 <= price <= 1000
  * 0 <= k < n

### The Bellman-Ford Algorithm (Layer-by-Layer)
Dijkstra's Algorithm finds the absolute cheapest path, but it doesn't care about the number of stops. Standard BFS finds the path with the fewest stops, but it doesn't care about the price. We need a hybrid!

Welcome to the **Bellman-Ford Algorithm**. Instead of a Min-Heap, this algorithm works in distinct "Waves" or "Layers".

**The Strategy:**
1. **The Price Board:** We create a board showing the cheapest known price to reach every airport. Initially, everything is `Infinity` except our starting airport (`src`), which is `0`.
2. **The Waves:** If we are allowed `k` stops, we can take at most `k + 1` flights. We will loop exactly `k + 1` times.
   * **Wave 1:** Find the cheapest prices using exactly 1 flight (0 stops).
   * **Wave 2:** Find the cheapest prices using up to 2 flights (1 stop).
   * **Wave 3:** Find the cheapest prices using up to 3 flights (2 stops).
3. **The Snapshot (Crucial Step):** Before every wave, we take a "snapshot" of the price board. When we calculate new prices, we only look at the snapshot. This guarantees we don't accidentally chain 5 flights together in a single wave!
4. **The Result:** After exactly `k + 1` waves, we look at the price board for our `dst`. If it is still `Infinity`, we couldn't reach it. Otherwise, that is our answer.

**The Math & Complexity Breakdown:**
* **Time Complexity:** $O(k \cdot E)$ — Where $E$ is the number of edges (flights). We loop through every single flight exactly $k + 1$ times.
* **Space Complexity:** $O(n)$ — We only need two arrays (the main array and the snapshot array) of size `n` to store the prices. We don't even need to build an Adjacency List!

In [9]:
from typing import List

class Solution:
    def findCheapestPrice(self, n: int, flights: List[List[int]], src: int, dst: int, k: int) -> int:
        # Step 1: Initialize the price board
        # Every city costs Infinity to reach initially
        prices = [float('inf')] * n
        prices[src] = 0
        
        # Step 2: Run Bellman-Ford for exactly (k + 1) waves
        # If k = 1 stop, we can take at most 2 flights.
        for i in range(k + 1):
            
            # Take a snapshot of the prices at the start of this wave
            tmp_prices = prices.copy()
            
            # Loop through every single flight on the map
            for u, v, price in flights:
                
                # If the starting airport 'u' is unreachable in our snapshot, skip it.
                if prices[u] == float('inf'):
                    continue
                    
                # If (Cost to reach 'u' + Flight from 'u' to 'v') is cheaper than 
                # our current known price to 'v', update the temp array!
                if prices[u] + price < tmp_prices[v]:
                    tmp_prices[v] = prices[u] + price
                    
            # After checking all flights, make the snapshot the new official board
            prices = tmp_prices
            
        # Step 3: Check the final price for our destination
        if prices[dst] == float('inf'):
            return -1
        else:
            return prices[dst]

### Problem 004: Reconstruct Flight Path (LeetCode 332) [HARD]

### Problem Definition and Constraints
You are given a list of flight `tickets` where `tickets[i] = [from_i, to_i]`. 
Your objective is to reconstruct the flight path that this person took, assuming they originally departed from `"JFK"` and used every single ticket exactly once.
If there are multiple valid flight paths, you must return the lexicographically smallest one (e.g., `"JFK" -> "ATL"` is chosen before `"JFK" -> "BOS"`).

**Examples:**
* **Example 1:**
  * **Input:** `tickets = [["BUF","HOU"],["HOU","SEA"],["JFK","BUF"]]`
  * **Output:** `["JFK","BUF","HOU","SEA"]`
* **Example 2:**
  * **Input:** `tickets = [["HOU","JFK"],["SEA","JFK"],["JFK","SEA"],["JFK","HOU"]]`
  * **Output:** `["JFK","HOU","JFK","SEA","JFK"]`

* Constraints:
  * 1 <= tickets.length <= 300
  * `from_i != to_i`
  * All tickets form at least one valid flight path.

### Core Logic: Eulerian Path (Hierholzer's Algorithm)
This problem asks us to use every *edge* (ticket) exactly once. In graph theory, visiting every edge exactly once is called an **Eulerian Path**. 

Because we want the lexicographically smallest path, our first instinct is standard backtracking: sort the destinations, try the smallest one, and if it leads to a dead end where we can't use the rest of our tickets, backtrack and try the next one. However, backtracking takes $O(E^V)$ time and is incredibly inefficient.

**Hierholzer's Algorithm (Post-Order DFS):**
Instead of backtracking, we can build the path flawlessly in $O(E)$ time. 
1. **The Dead End Secret:** In a valid Eulerian path, you can only ever get permanently "stuck" (reach a node with no unused outgoing tickets) if you have physically reached the **absolute final destination** of the entire journey. 
2. **Post-Order Construction:** If we perform a DFS, and we blindly pop tickets and fly to the next airport, the very first time we hit a dead end, we are guaranteed to be at the final stop. We write that airport down in our result array. As the DFS naturally unwinds (backtracks up the call stack), we write down the airports as we retreat. 
3. **The Reversal:** Because we hit the final destination first and wrote it down, our result array contains the journey in perfect **reverse order**. We just reverse the array at the end!

To handle the lexicographical requirement efficiently, we sort the initial tickets in *reverse* alphabetical order. This allows us to use $O(1)$ `.pop()` operations on the adjacency list to always grab the alphabetically smallest destination.

### Approach: Adjacency List + Post-Order DFS
1. Sort the `tickets` array in descending (reverse-lexicographical) order.
2. Build an adjacency list `adj`. Because of the reverse sorting, `adj["JFK"]` might look like `["SFO", "ATL"]`. 
3. Initialize a `res` array.
4. Create a `dfs(airport)` function:
   * While `adj[airport]` still has tickets (edges) left:
     * `pop()` the last ticket (which is the lexicographically smallest).
     * Recursively call `dfs()` on that newly popped destination.
   * When the `while` loop finishes (or if the airport has no outgoing flights left), append the `airport` to `res`.
5. Call `dfs("JFK")`.
6. Return `res[::-1]` (the reversed result array).

* **Time Complexity:** $O(E \log E)$ — Where $E$ is the number of tickets. Sorting the tickets takes $O(E \log E)$. The DFS takes strictly $O(E)$ time because we physically `pop` and consume each edge exactly once. 
* **Space Complexity:** $O(E)$ — The adjacency list and the recursive call stack will scale directly with the number of tickets.

In [10]:
import collections
from typing import List

class Solution:
    def findItinerary(self, tickets: List[List[str]]) -> List[str]:
        # 1. Sort the tickets in reverse lexicographical order
        # This ensures that when we pop() from the end of a list, 
        # we get the alphabetically smallest airport in O(1) time.
        tickets.sort(reverse=True)
        
        # 2. Build the Adjacency List
        adj = collections.defaultdict(list)
        for src, dst in tickets:
            adj[src].append(dst)
            
        res = []
        
        # 3. Hierholzer's Algorithm (Post-Order DFS)
        def dfs(airport):
            # While this airport still has outgoing tickets left
            while adj[airport]:
                # Grab the alphabetically smallest destination and consume the ticket
                next_dest = adj[airport].pop()
                
                # Fly to that destination
                dfs(next_dest)
                
            # If we break out of the while loop, we are out of outgoing tickets.
            # We have hit a dead end. In an Eulerian path, this is the final stop!
            # Append it to the result as we unwind the recursion.
            res.append(airport)
            
        # The journey always begins at JFK
        dfs("JFK")
        
        # The path was built backward from the final destination, so reverse it
        return res[::-1]


        """
        =========================================================
        SIMULATION NOTES: 
        tickets = [["JFK","KUL"], ["JFK","NRT"], ["NRT","JFK"]]
        =========================================================
        Sort reverse: [["NRT","JFK"], ["JFK","NRT"], ["JFK","KUL"]]
        adj = {
          "JFK": ["NRT", "KUL"],
          "NRT": ["JFK"]
        }
        
        dfs("JFK"):
          adj["JFK"] pops "KUL" (smallest alphabetically).
          
          dfs("KUL"):
            adj["KUL"] is empty! Hit a dead end.
            res.append("KUL")  --> res = ["KUL"]
            return to JFK.
            
          adj["JFK"] pops "NRT".
          
          dfs("NRT"):
            adj["NRT"] pops "JFK".
            
            dfs("JFK"):
              adj["JFK"] is empty! 
              res.append("JFK") --> res = ["KUL", "JFK"]
              return to NRT.
              
            res.append("NRT") --> res = ["KUL", "JFK", "NRT"]
            return to JFK.
            
          res.append("JFK") --> res = ["KUL", "JFK", "NRT", "JFK"]
          
        Reverse res: ["JFK", "NRT", "JFK", "KUL"]
        """

### Problem 005: Swim in Rising Water (LeetCode 778) [HARD]

### Problem Definition and Constraints
You are given a square 2-D matrix of distinct integers `grid` where each integer `grid[i][j]` represents the elevation at position `(i, j)`.
Rain starts to fall at time `t = 0`, causing the water level to rise uniformly. At time `t`, the water level across the entire grid is `t`.
You may swim either horizontally or vertically to an adjacent square if the original elevation of both squares is less than or equal to the water level at time `t`.
Starting from the top-left square `(0, 0)`, return the minimum amount of time it will take until it is possible to reach the bottom-right square `(n - 1, n - 1)`.

**Examples:**
* **Example 1:**
  * **Input:** `grid = [[0,1], [2,3]]`
  * **Output:** `3`
  * **Explanation:** You start at `(0,0)` with elevation 0. You cannot move to `(1,0)` until `t = 2`, but you can move to `(0,1)` at `t = 1`. However, to reach `(1,1)` from either path, you must wait until `t = 3`. 
* **Example 2:**
  * **Input:** `grid = [[0,1,2,10], [9,14,4,13], [12,3,8,15], [11,5,7,6]]`
  * **Output:** `8`
  * **Explanation:** The optimal path snakes through the center to avoid the `14`, `13`, and `15` elevations. The highest peak on this optimal path is `8`, so you must wait until `t = 8` to complete the swim.

* Constraints:
  * `grid.length == grid[i].length == n`
  * 1 <= n <= 50
  * $0 \le \text{grid}[i][j] < n^2$

### Core Logic: Dijkstra's Algorithm (The "Bottleneck" Path)
This is a variation of the Shortest Path problem. However, instead of the cost being the *sum* of the edges along the path, the cost of a path is strictly defined by its **bottleneck**—the single highest elevation you must cross.

We use **Dijkstra's Algorithm** with a Min-Heap to continuously expand our search through the lowest possible bottlenecks.
1. Our Min-Heap stores tuples of `(max_elevation_so_far, r, c)`.
2. Because it is a Min-Heap, it guarantees that we always process the path with the lowest peak first. 
3. When we stand on a cell and look at its neighbor, the new bottleneck for that path becomes `max(current_bottleneck, neighbor_elevation)`.
4. The exact moment we pop the destination `(n - 1, n - 1)` from the heap, we are mathematically guaranteed that the `max_elevation_so_far` attached to it is the absolute lowest possible bottleneck across all possible paths.

### Approach: Min-Heap Grid Traversal
1. Initialize a `min_heap` with the starting cell: `[(grid[0][0], 0, 0)]`.
2. Initialize a `visited` set to prevent walking in circles, starting with `(0, 0)`.
3. Pop the cell with the lowest `time` (bottleneck) from the heap.
4. If the popped cell is `(n - 1, n - 1)`, immediately return `time`.
5. Iterate through all 4 valid neighbors. If a neighbor has not been visited:
   * Calculate the new bottleneck: `new_time = max(time, grid[nr][nc])`.
   * Mark it as visited and push `(new_time, nr, nc)` to the heap.

* **Time Complexity:** $O(n^2 \log n)$ — There are $n^2$ total cells. In the worst case, we push all of them into the Min-Heap. Pushing to a heap of size $n^2$ takes $O(\log(n^2))$ which simplifies to $O(\log n)$.
* **Space Complexity:** $O(n^2)$ — The `visited` set and the `min_heap` will store up to $n^2$ cells.

In [11]:
import heapq
from typing import List

class Solution:
    def swimInWater(self, grid: List[List[int]]) -> int:
        n = len(grid)
        
        # Min-Heap stores (max_time_encountered_on_path, row, col)
        min_heap = [(grid[0][0], 0, 0)]
        visited = set([(0, 0)])
        
        directions = [[0, 1], [0, -1], [1, 0], [-1, 0]]
        
        while min_heap:
            # Pop the path with the lowest bottleneck so far
            t, r, c = heapq.heappop(min_heap)
            
            # If we reached the bottom-right corner, we are done
            if r == n - 1 and c == n - 1:
                return t
                
            # Explore all 4 neighbors
            for dr, dc in directions:
                nr, nc = r + dr, c + dc
                
                # Check bounds and visited status
                if (nr < 0 or nr >= n or 
                    nc < 0 or nc >= n or 
                    (nr, nc) in visited):
                    continue
                    
                # Mark as visited instantly to prevent duplicate heap additions
                visited.add((nr, nc))
                
                # The time required to reach the neighbor is bounded by the HIGHEST 
                # elevation encountered so far.
                new_t = max(t, grid[nr][nc])
                
                heapq.heappush(min_heap, (new_t, nr, nc))


        """
        =========================================================
        SIMULATION NOTES: 
        grid = [
          [0, 2],
          [1, 3]
        ]
        =========================================================
        
        Start: min_heap = [(0, 0, 0)], visited = {(0, 0)}
        
        Pop (0, 0, 0). 
        Neighbors of (0,0):
        - (0, 1): height 2. max(0, 2) = 2. Push (2, 0, 1). visited={(0,0), (0,1)}
        - (1, 0): height 1. max(0, 1) = 1. Push (1, 1, 0). visited={(0,0), (0,1), (1,0)}
        
        min_heap is now [(1, 1, 0), (2, 0, 1)].
        
        Pop (1, 1, 0) because 1 < 2!
        Neighbors of (1,0):
        - (1, 1): height 3. max(1, 3) = 3. Push (3, 1, 1). visited={(0,0), (0,1), (1,0), (1,1)}
        
        min_heap is now [(2, 0, 1), (3, 1, 1)].
        
        Pop (2, 0, 1) because 2 < 3!
        Neighbors of (0, 1):
        - (1, 1) is already visited! Skip.
        
        min_heap is now [(3, 1, 1)].
        
        Pop (3, 1, 1). 
        r == 1 and c == 1! Return 3.
        
        Notice how Dijkstra's naturally paused exploring the path through the '1' 
        because the next step was a '3', allowing it to evaluate the path through 
        the '2' first to guarantee the absolute minimum bottleneck.
        """

### Problem 006: Alien Dictionary (LeetCode 269) [HARD]

### Problem Definition and Constraints
You are given a list of strings `words` representing a dictionary in a new alien language. The words are sorted lexicographically by the rules of this language.
Your goal is to derive the alphabetical order of the alien language and return it as a string of unique letters.
If the given arrangement is mathematically impossible, return `""`. If there are multiple valid solutions, return any of them.

**Examples:**
* **Example 1:**
  * **Input:** `words = ["z","o"]`
  * **Output:** `"zo"`
* **Example 2:**
  * **Input:** `words = ["hrn","hrf","er","enn","rfnn"]`
  * **Output:** `"hernf"`
* **Example 3:**
  * **Input:** `words = ["abc","ab"]`
  * **Output:** `""`
  * **Explanation:** "ab" is a prefix of "abc". In any valid dictionary, a prefix must appear *before* the longer word. Since "abc" comes first, the input is invalid.

* Constraints:
  * 1 <= words.length <= 100
  * 1 <= words[i].length <= 100
  * Only lowercase English letters.

### Core Logic: Topological Sort (3-State DFS)
This is the ultimate test of Directed Graphs and Topological Sorting. We extract the rules of the alien language by comparing adjacent words.
If we compare "ape" and "apple", the first letter that differs is 'e' and 'p'. Because "ape" comes first, we know definitively that `e < p`. This represents a directed edge `e -> p`.

**The Three Phases:**
1. **The Prefix Trap:** Before building edges, we must check for the invalid prefix rule (Example 3). If `word1` is longer than `word2` but starts with `word2`, we immediately return `""`.
2. **Build the Graph:** We compare every adjacent pair of words. We find the *first* character that differs, draw a directed edge from `char1` to `char2`, and instantly break the loop (subsequent characters in those words give us no valid alphabetical information).
3. **Topological Sort:** We use the exact 3-State DFS from Course Schedule II to traverse the characters. 
   * `False` (Unvisited): We haven't seen this letter yet.
   * `True` (Visiting): This letter is currently in our active path. If we hit it again, we found a cycle (e.g., `a < b` and `b < a`). Return `""`.
   * (Removed from path, added to result): The letter and all its descendants are fully processed.

### Approach: Adjacency List + Post-Order DFS
1. Initialize an adjacency list `adj` containing every unique character across all words. (If a character exists in the words but has no edges, it still needs to be in our output).
2. Iterate through pairs of adjacent words (`w1` and `w2`). 
   * Check the prefix trap.
   * Zip through their characters. At the first mismatch, do `adj[char1].add(char2)` and break.
3. Initialize a `visited` dictionary to track the 3 states (Node -> Boolean: `True` = in active path, `False` = fully visited and safe).
4. Run the DFS on every unique character.
   * If a cycle is detected, return `""`.
   * Post-order traversal: append the character to `res` *after* exploring its neighbors.
5. Because post-order builds the list backward (the "Z" of the alphabet gets added first), reverse the `res` list and join it into a string.

* **Time Complexity:** $O(C)$ — Where $C$ is the total number of characters across all words. Building the graph takes $O(C)$ time. The DFS takes $O(V + E)$ time, but since $V \le 26$ and $E \le 26^2$, the graph traversal is strictly $O(1)$ relative to the input size.
* **Space Complexity:** $O(1)$ — The adjacency list, visited map, and result string will never hold more than 26 characters.

In [12]:
from typing import List

class Solution:
    def alienOrder(self, words: List[str]) -> str:
        # 1. Initialize Adjacency List with EVERY unique character
        # We use a set for neighbors to prevent duplicate edges
        adj = {char: set() for word in words for char in word}
        
        # 2. Build the Graph
        for i in range(len(words) - 1):
            w1, w2 = words[i], words[i + 1]
            min_len = min(len(w1), len(w2))
            
            # THE PREFIX TRAP: e.g., ["abc", "ab"] is mathematically invalid
            if len(w1) > len(w2) and w1[:min_len] == w2[:min_len]:
                return ""
                
            # Find the first character difference to build the edge
            for j in range(min_len):
                if w1[j] != w2[j]:
                    adj[w1[j]].add(w2[j])
                    # Stop after the first difference! Subsequent letters tell us nothing.
                    break
                    
        # 3. Topological Sort (3-State DFS)
        # visited maps char -> boolean
        # True = currently in the active path (Visiting)
        # False = completely processed and safe (Visited)
        visited = {}
        res = []
        
        def dfs(char):
            # BASE CASE 1: Cycle Detected!
            if char in visited:
                return visited[char]
                
            # CHOOSE: Mark as currently in the active timeline
            visited[char] = True
            
            # EXPLORE: Dive into all characters that come AFTER this one
            for neighbor in adj[char]:
                # If any downstream neighbor finds a cycle, bubble up the failure
                if dfs(neighbor):
                    return True
                    
            # UNDO & SUCCESS: We hit the absolute bottom of this chain.
            # Mark it as permanently safe (False), and add it to the result.
            visited[char] = False
            res.append(char)
            
            return False

        # 4. The Manager Loop
        # Check every single unique character in our dictionary
        for char in adj:
            # If the DFS returns True, a cycle was found.
            if dfs(char):
                return ""
                
        # 5. Reverse the post-order result to get the true chronological order
        return "".join(res[::-1])


        """
        =========================================================
        SIMULATION NOTES: words = ["wrt", "wrf", "er", "ett", "rftt"]
        =========================================================
        
        Graph Building:
        "wrt" vs "wrf" -> 't' != 'f' -> adj['t'].add('f')
        "wrf" vs "er"  -> 'w' != 'e' -> adj['w'].add('e')
        "er" vs "ett"  -> 'r' != 't' -> adj['r'].add('t')
        "ett" vs "rftt" -> 'e' != 'r' -> adj['e'].add('r')
        
        Adjacency List:
        t -> {f}
        w -> {e}
        r -> {t}
        e -> {r}
        
        DFS Execution (Starting from 'w'):
        dfs('w'):
          neighbors of 'w' -> 'e'
          dfs('e'):
            neighbors of 'e' -> 'r'
            dfs('r'):
              neighbors of 'r' -> 't'
              dfs('t'):
                neighbors of 't' -> 'f'
                dfs('f'):
                  no neighbors.
                  visited['f'] = False. res.append('f')
                visited['t'] = False. res.append('t')
              visited['r'] = False. res.append('r')
            visited['e'] = False. res.append('e')
          visited['w'] = False. res.append('w')
          
        res = ['f', 't', 'r', 'e', 'w']
        Reversed: "wertf"
        """